In [1]:
# Cell 0: Master Protocol Generation (With Physical Verification & Missing File Tracking)
import os
import pandas as pd
import random

# ==============================================================================
# 1. Path Configuration
# ==============================================================================
BASE_WF = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/WaveFake"
BASE_ASV = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/LA"
BASE_WILD = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/In_The_Wild"
OUTPUT_DIR = "./protocol"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def build_protocol():
    """
    Consolidates multiple datasets with strict physical file verification.
    """
    master_list = []
    # 跟踪各个来源丢失的文件数量
    missing_tracker = {"WF": 0, "ASV": 0, "Wild": 0}

    # --- Utility: Safe Add with Double Check ---
    def safe_add(path, label, model_name, source):
        if os.path.exists(path):
            master_list.append({
                "path": path, "label": label, 
                "model_name": model_name, "source": source
            })
            return True
        return False

    def get_files(path, ext=(".wav", ".WAV", ".flac")):
        if not os.path.exists(path): return []
        return [os.path.join(path, f) for f in os.listdir(path) if f.lower().endswith(ext)]

    # ---- 1. LJSpeech Real ----
    ljs_real_path = os.path.join(BASE_WF, "LJSpeech-1.1_real")
    ljs_real_files = get_files(ljs_real_path)
    count = 0
    for f in ljs_real_files:
        if safe_add(f, 1, "original", "LJ_Real"): count += 1
    print(f"-> LJ Real Samples found: {count}")

    # ---- 2. WaveFake LJ Fake ----
    fake_folders = [
        "ljspeech_full_band_melgan", "ljspeech_hifiGAN", "ljspeech_melgan",
        "ljspeech_melgan_large", "ljspeech_multi_band_melgan",
        "ljspeech_parallel_wavegan", "ljspeech_waveglow"
    ]
    for i, folder in enumerate(fake_folders):
        f_path = os.path.join(BASE_WF, folder)
        f_files = sorted(get_files(f_path))
        sample = f_files[i*1872 : (i+1)*1872]
        model_tag = folder.split('_')[-1]
        for f in sample:
            if not safe_add(f, 0, model_tag, "WF_LJ"):
                missing_tracker["WF"] += 1
    print(f"-> Total after WaveFake LJ integration: {len(master_list)}")

    # ---- 3. JSUT Dataset ----
    jsut_real_files = get_files(os.path.join(BASE_WF, "jsut_real"))
    count_jsut_r = 0
    for f in jsut_real_files:
        if safe_add(f, 1, "original", "JSUT_Real"): count_jsut_r += 1
    print(f"-> JSUT Real Samples found: {count_jsut_r}")

    for folder in ["jsut_multi_band_melgan", "jsut_parallel_wavegan"]:
        f_files = get_files(os.path.join(BASE_WF, folder))
        for f in f_files:
            if not safe_add(f, 0, folder.split('_')[-1], "JSUT_Fake"):
                missing_tracker["WF"] += 1

    # ---- 4. Common Voice (WF_CV) ----
    cv_path = os.path.join(BASE_WF, "common_voices_prompts_from_conformer_fastspeech2_pwg_ljspeech")
    cv_files = get_files(cv_path)
    count_cv = 0
    for f in cv_files:
        if safe_add(f, 0, "fastspeech2_pwg", "WF_CV"): count_cv += 1
        else: missing_tracker["WF"] += 1
    print(f"-> Successfully added ALL {count_cv} samples from Common Voice (WF_CV).")

    # ---- 5. In-the-wild Data ----
    wild_csv = os.path.join(BASE_WILD, "meta.csv")
    if os.path.exists(wild_csv):
        df_wild = pd.read_csv(wild_csv, sep=None, engine='python')
        for _, row in df_wild.iterrows():
            f_path = os.path.join(BASE_WILD, str(row['file']).strip())
            if not safe_add(f_path, 1 if row['label'] == 'bona-fide' else 0, "unknown_wild", "Wild"):
                missing_tracker["Wild"] += 1

    # ---- 6. ASVspoof 2019 (Verified Balancing) ----
    asv_prot = os.path.join(BASE_ASV, "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt")
    if os.path.exists(asv_prot):
        asv_data = []
        with open(asv_prot, "r") as f:
            for line in f:
                parts = line.strip().split()
                path = os.path.join(BASE_ASV, "ASVspoof2019_LA_train/flac", parts[1] + ".flac")
                if os.path.exists(path):
                    asv_data.append({
                        "path": path, "label": 1 if parts[4] == "bonafide" else 0, 
                        "model_name": parts[3] if parts[4] == "spoof" else "original", 
                        "source": "ASV"
                    })
                else:
                    missing_tracker["ASV"] += 1
        df_asv = pd.DataFrame(asv_data)
        if not df_asv.empty:
            reals = df_asv[df_asv['label'] == 1]
            fakes = df_asv[df_asv['label'] == 0].sample(n=min(len(reals), len(df_asv[df_asv['label']==0])), random_state=42) 
            master_list.extend(reals.to_dict('records'))
            master_list.extend(fakes.to_dict('records'))

    # ---- 7. Final Export & Pretty Report ----
    df_final = pd.DataFrame(master_list)
    final_path = os.path.join(OUTPUT_DIR, "master_protocol.csv")
    df_final.to_csv(final_path, index=False)

    print("\n" + "="*50)
    print("📋 FINAL DATASET GENERATION REPORT")
    print("="*50)
    print(f"{'Total Samples':<18} : {len(df_final)}")
    print(f"{'Real (Label 1)':<18} : {len(df_final[df_final['label']==1])}")
    print(f"{'Fake (Label 0)':<18} : {len(df_final[df_final['label']==0])}")
    
    # 打印丢失文件报告（如果存在丢失的话）
    has_missing = any(v > 0 for v in missing_tracker.values())
    if has_missing:
        print("-" * 50)
        print("⚠️  MISSING FILES DETECTED (Skipped):")
        for src, val in missing_tracker.items():
            if val > 0:
                print(f"   - {src:<15} : {val} files")
                
    print("-" * 50)
    print("Distribution by Source:")
    print(df_final['source'].value_counts())
    print("="*50)
    print(f"✅ Success: Master protocol saved to {final_path}")

if __name__ == "__main__":
    build_protocol()

-> LJ Real Samples found: 13100
-> Total after WaveFake LJ integration: 26200
-> JSUT Real Samples found: 5000
-> Successfully added ALL 16283 samples from Common Voice (WF_CV).

📋 FINAL DATASET GENERATION REPORT
Total Samples      : 94422
Real (Label 1)     : 40643
Fake (Label 0)     : 53779
--------------------------------------------------
Distribution by Source:
source
Wild         31779
WF_CV        16283
LJ_Real      13100
WF_LJ        13100
JSUT_Fake    10000
ASV           5160
JSUT_Real     5000
Name: count, dtype: int64
✅ Success: Master protocol saved to ./protocol/master_protocol.csv
